# Stage 2 final TEST evaluation (IE643 floor-plan completion) - run once

What this notebook does (everything from **Save Version -> Save & Run All**, no API token, no internet needed):

1. copies the project code from the attached dataset `floorplan-code` and installs two packages from the wheels in
   `floorplan-stage2-final` (offline);
2. puts the checkpoints from `floorplan-stage2-final` where the code expects them;
3. finds the CubiCasa **test** split (`parser_test.zip`, `completion_test.zip`) in the attached inputs;
4. runs `src/stage2/final_test.py` as **two processes, one per T4 GPU**, cases split by index; every finished case is
   written at once to `/kaggle/working/final_test/part<k>.jsonl` (checkpoint): if the run is cut off, re-run with this
   notebook's previous output attached and it continues where it stopped;
5. writes the report `/kaggle/working/final_test/final_test.md` (fill ablation constrained vs control, CHECK on test
   drawings, end to end, all by omission type / location / position / size bin) and zips the folder.

Settings: GPU **T4 x2**; Internet may stay off. Expected time: about 3-5 h.

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
WORK = "/kaggle/working"
CODE = "/tmp/code"
OUT = f"{WORK}/final_test"
os.makedirs(OUT, exist_ok=True)

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

# 1. code (the floorplan-code dataset holds code.zip, or its unpacked folder)
hits = [h for h in find("src/utils.py") if "floorplan-code" in h]
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE, dirs_exist_ok=True)
else:
    zipfile.ZipFile([h for h in find("code.zip") if "floorplan-code" in h][0]).extractall(CODE)
print("code ->", CODE, os.listdir(CODE)[:10])

In [ ]:
# 1b. packages, offline from the wheels in floorplan-stage2-final (skipped if already installed)
wheels = [os.path.dirname(w) for w in find("wheels/*.whl") + find("wheels/*.tar.gz")]
def have(mod):
    try:
        __import__(mod)
        return True
    except ImportError:
        return False
for mod, pkg in (("segmentation_models_pytorch", "segmentation-models-pytorch==0.3.4"), ("diffusers", "diffusers")):
    if not have(mod):
        args = [sys.executable, "-m", "pip", "install", "-q", pkg]
        if wheels:
            args += ["--no-index", "--find-links", wheels[0]]
        print(" ".join(args)); subprocess.run(args, check=True)
print("smp", have("segmentation_models_pytorch"), "diffusers", have("diffusers"))

In [ ]:
# 2. checkpoints: floorplan-stage2-final/checkpoints/<path> -> CODE/<path> (paths of configs/stage2/pipeline.yaml)
import yaml
paths = yaml.safe_load(open(f"{CODE}/configs/stage2/pipeline.yaml", encoding="utf-8"))["checkpoints"]
roots = [h[: -len("checkpoints/" + p)] for name, p in paths.items() for h in find("checkpoints/" + p)]
assert roots, "floorplan-stage2-final (checkpoints) is not attached"
root = roots[0]
for name, p in paths.items():
    src = os.path.join(root, "checkpoints", p)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(f"{CODE}/{p}"), exist_ok=True)
        shutil.copy(src, f"{CODE}/{p}")
        print(f"{name:18s} ok")
    else:
        print(f"{name:18s} MISSING", src)

# 3. test data
PZ, CZ = find("parser_test.zip"), find("completion_test.zip")
assert PZ and CZ, "parser_test.zip / completion_test.zip not found: attach the floorplan-stage2-data output"
PZ, CZ = PZ[0], CZ[0]
print(PZ, CZ)

In [ ]:
# 4a. resume: copy the finished cases of an earlier version of this notebook (if its output is attached)
for f in find("final_test/part*.jsonl"):
    dst = f"{OUT}/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print("resuming from", f, sum(1 for _ in open(f)), "cases")

# 4b. quick GPU check (no test data touched)
import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "set Accelerator = GPU T4 x2"

In [ ]:
# 4c. the evaluation: one process per GPU
n = max(1, torch.cuda.device_count())
env = {**os.environ, "PYTHONUTF8": "1", "PYTHONUNBUFFERED": "1", "STAGE2_DEVICE": "cuda", "PYTHONPATH": CODE}
procs = []
for k in range(n):
    log = open(f"{OUT}/part{k}.log", "a")
    cmd = [sys.executable, "-m", "src.stage2.final_test", "--parser-zip", PZ, "--completion-zip", CZ, "--out", OUT,
           "--part", str(k), "--parts", str(n)]
    procs.append(subprocess.Popen(cmd, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                                  env={**env, "CUDA_VISIBLE_DEVICES": str(k)}))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    done = sum(sum(1 for _ in open(f)) for f in glob.glob(f"{OUT}/part*.jsonl"))
    print(f"{(time.time() - t0) / 3600:.2f} h, {done} cases done", flush=True)
codes = [p.wait() for p in procs]
print("exit codes", codes)
for k in range(n):
    print(f"===== part{k}.log (tail)"); print("".join(open(f"{OUT}/part{k}.log").readlines()[-20:]))

In [ ]:
# 5. report + zip of all outputs
subprocess.run([sys.executable, "-m", "src.stage2.final_test", "--report", "--out", OUT], cwd=CODE,
               env={**env, "STAGE2_DEVICE": "cpu"}, check=True)
shutil.make_archive(f"{WORK}/final_test", "zip", OUT)
print(open(f"{OUT}/final_test.md", encoding="utf-8").read())